# Graph storage adapters

All three adapters implement `BaseGraphStorage` and take the same arguments (a
location plus the `node_cls` / `edge_cls` used to materialize records), so `Index`
can swap them without any pipeline change. What differs is the operational
envelope:

- **`NetworkXStorage`** — in-process `MultiDiGraph` serialized to a `.gml` file. The
  default. No server, but the whole graph lives in RAM and `index_done_callback()`
  rewrites the entire file.
- **`Neo4jStorage`** — client/server, the option that scales past a single process.
  Deliberately not re-exported from `ragu.storage`, because importing it would make
  the `neo4j` driver a hard dependency for everyone.

RAGU graphs are directed **multigraphs**: a pair of nodes may hold several edges.
That is why edges are addressed by `EdgeSpec` tuples
`(subject_id, object_id, relation_id)` and why `get_edges` returns one list per
spec. A `relation_id` of `None` means "every edge between this pair".

This notebook needs **no API keys** — the first two adapters run entirely locally.

In [ ]:
import os
import shutil
import tempfile
from pathlib import Path

from ragu.graph.types import Entity, Relation
from ragu.storage.base_storage import BaseGraphStorage
from ragu.storage.graph_storage_adapters import NetworkXStorage

## A tiny graph

Two of the relations connect the same pair of nodes. That is legal here, and it is
the reason edges carry their own ids.

In [7]:
RITCHIE = Entity(
    entity_name="Dennis Ritchie",
    entity_type="PERSON",
    description="Creator of the C programming language.",
    source_chunk_id=["chunk-1"],
)
BELL_LABS = Entity(
    entity_name="Bell Labs",
    entity_type="ORGANIZATION",
    description="Research laboratory where C and Unix were developed.",
    source_chunk_id=["chunk-1", "chunk-2"],
)
UNIX = Entity(
    entity_name="Unix",
    entity_type="PRODUCT",
    description="Operating system developed at Bell Labs.",
    source_chunk_id=["chunk-2"],
)
ENTITIES = [RITCHIE, BELL_LABS, UNIX]

RELATIONS = [
    Relation(
        subject_id=RITCHIE.id,
        object_id=BELL_LABS.id,
        subject_name=RITCHIE.entity_name,
        object_name=BELL_LABS.entity_name,
        relation_type="WORKS_AS",
        description="Dennis Ritchie worked at Bell Labs.",
        source_chunk_id=["chunk-1"],
    ),
    Relation(
        subject_id=RITCHIE.id,
        object_id=BELL_LABS.id,
        subject_name=RITCHIE.entity_name,
        object_name=BELL_LABS.entity_name,
        relation_type="MEMBER_OF",
        description="Dennis Ritchie was a member of the Bell Labs research staff.",
        source_chunk_id=["chunk-1"],
    ),
    Relation(
        subject_id=UNIX.id,
        object_id=BELL_LABS.id,
        subject_name=UNIX.entity_name,
        object_name=BELL_LABS.entity_name,
        relation_type="CREATED_BY",
        description="Unix was created at Bell Labs.",
        source_chunk_id=["chunk-2"],
    ),
]

## One scenario, any backend

The whole point of the base class: this function never learns which adapter it got.

In [8]:
async def exercise(label: str, storage: BaseGraphStorage) -> None:
    """
    Run the same sequence of graph operations against any adapter.

    :param label: Adapter name used in the printed output.
    :param storage: Adapter instance to exercise.
    """
    print("=" * 70)
    print(label)
    print("=" * 70)

    # Surfaces connection problems here rather than inside the first write.
    await storage.index_start_callback()

    await storage.upsert_nodes(ENTITIES)
    await storage.upsert_edges(RELATIONS)
    # NetworkX writes its GML file here; the database-backed adapters no-op.
    await storage.index_done_callback()

    print(f"\nnodes: {len(await storage.get_all_nodes())}, edges: {len(await storage.get_all_edges())}")

    # Reads are index-aligned: a missing id yields None in its own slot rather than
    # shifting the results.
    fetched = await storage.get_nodes([RITCHIE.id, "ent-missing"])
    print(f"get_nodes: {[node.entity_name if node else None for node in fetched]}")

    # relation_id=None matches every edge between the pair.
    between = (await storage.get_edges([(RITCHIE.id, BELL_LABS.id, None)]))[0]
    print(f"\nedges between Ritchie and Bell Labs: {len(between)}")
    for edge in between:
        print(f"  {edge.relation_type}: {edge.description}")

    named = (await storage.get_edges([(RITCHIE.id, BELL_LABS.id, RELATIONS[0].id)]))[0]
    print(f"edges matching an explicit relation id: {len(named)}")

    # Degree is subject degree + object degree, and it is what local search uses to
    # rank relations by how central they are.
    specs = [(edge.subject_id, edge.object_id, edge.id) for edge in RELATIONS]
    for edge, degree in zip(RELATIONS, await storage.edges_degrees(specs)):
        print(f"  degree({edge.relation_type}) = {degree}")

    incident = await storage.get_all_edges_for_nodes([BELL_LABS.id, UNIX.id])
    print(f"\nincident edges: Bell Labs={len(incident[0])}, Unix={len(incident[1])}")

    await storage.delete_edges([(UNIX.id, BELL_LABS.id, None)])
    await storage.delete_nodes([UNIX.id])
    await storage.index_done_callback()
    print(f"after deleting Unix: {len(await storage.get_all_nodes())} nodes, "
          f"{len(await storage.get_all_edges())} edges")

    # Mandatory for connection- and WAL-backed adapters, no-op for NetworkX.
    await storage.close()

In [9]:
workdir = Path(tempfile.mkdtemp(prefix="ragu_graph_adapters_"))

## NetworkX — in-memory MultiDiGraph, GML on disk

In [ ]:
await exercise(
    "NetworkXStorage",
    NetworkXStorage(
        filename=str(workdir / "graph.gml"),
        node_cls=Entity,
        edge_cls=Relation,
    ),
)

In [ ]:
# Neo4j below does not use the working directory, so clean it up here rather than
# at the end of the notebook — that cell needs a running server and may not run.
shutil.rmtree(workdir, ignore_errors=True)
print(f"removed {workdir}")

## Neo4j — client/server

Needs a running server, so this cell is the only one in the notebook that will not
work out of the box:

```
docker run -d -p 7687:7687 -e NEO4J_AUTH=neo4j/password neo4j:5
```

The import is by full path on purpose: `ragu.storage` does not re-export
`Neo4jStorage`, so the `neo4j` driver stays optional.

In [ ]:
from ragu.storage.graph_storage_adapters.neo4j_adapter import Neo4jStorage

await exercise(
    "Neo4jStorage",
    Neo4jStorage(
        uri=os.environ.get("NEO4J_URI", "bolt://localhost:7687"),
        user=os.environ.get("NEO4J_USER", "neo4j"),
        password=os.environ.get("NEO4J_PASSWORD", "password"),
        node_cls=Entity,
        edge_cls=Relation,
    ),
)